# 02 — Thu gom toàn bộ nội dung BTC từ AI Guru (Colab CPU)

**Chọn CPU → Run all.** Không chạy lại 02/03 cũ, không cần GPU hay Kaggle token.
Quét toàn bộ folder AI Guru qua Drive API, gồm shard cũ, mới, mọi bản resume,
các TAR lồng nhau và retained. Mỗi URL được đối chiếu với snapshot BTC
`71be0e12`; ID và URL lấy nguyên từ file BTC. Không đếm file/thành công HTTP
như tài liệu hợp lệ. Giữ riêng snapshot gốc của crawl/extraction.

Dùng lại mapping baseline và 10 nguồn đã freeze để tránh xử lý lại tài liệu.
Chỉ lưu thêm nội dung vượt kiểm tra metadata/extraction vào kho recovery trên Drive.
Không copy nguyên TAR/raw HTML, không encode model, không rebuild index hay đổi
candidate của luồng submit. Dữ liệu tạm chỉ ở runtime Colab; không tải về PC.

Checkpoint sau mỗi leaf/archive gồm một SQL snapshot nén trên Drive. Đổi VM
phục hồi một snapshot; archive/mapping đã commit không mở lại. Một leaf ngắt
giữa chừng có thể đọc lại leaf đó. Không tự xóa checkpoint hoặc archive gốc.

**Đã sửa lỗi giới hạn 128 header:** native leaf đọc đủ bốn manifest thì
dừng liệt kê raw. Container vẫn duyệt hết mọi TAR con, gồm resume ở cuối.
Nếu bản trước báo 26 archive lỗi, mở bản này và Run all trên CPU; mapping
và archive đã commit được bỏ qua, chỉ các archive lỗi được thử tiếp.
Không xóa folder checkpoint và không chạy đồng thời hai phiên.

Đầu ra `data_staging/btc-all-ai-guru-71be0e12-v1`:
`collection_report.json`, `btc_membership.csv`, `missing_or_unusable.csv`,
các Parquet chứa text recovery cùng nguồn gốc raw/hash/snapshot.
`ALL_FOLDER_SOURCES_COLLECTED` chỉ xác nhận đọc xong folder; xem riêng
`all_official_content_available` và các nhóm trạng thái để biết có đủ nội dung.
Nội dung recovery vẫn cần chunk/freeze chuẩn; đây chưa phải pipeline inference
hay xác nhận mọi URL trên Internet đều truy cập được.
Chỉ chạy một phiên CPU notebook này trên cùng output folder.

In [ ]:
#@title 1. Colab CPU — toàn bộ AI Guru, không chỉ batch 10 nguồn
import importlib
import json
import os
from pathlib import Path
import subprocess
import sys
print('ALL BTC CONTENT: read metadata across all archive versions, retain existing checkpoints.',flush=True)
if not Path('/var/colab/hostname').is_file():
    raise RuntimeError('Chạy notebook này trên Colab CPU; không dùng GPU/Kaggle cho bước thu gom Drive.')
from google.colab import drive,auth
drive.mount('/content/drive')
PROJECT=Path('/content/drive/MyDrive/VietMedBridge')
parent='data_staging/btc-new-71be0e12-20261019-v1'
if not (PROJECT/parent/'full_intake_scope.json').is_file():
    PROJECT=Path('/content/drive/.shortcut-targets-by-id/1uKKHYEcMPodnJIscKPi6jsoNBJvyhFGK/VietMedBridge')
PROJECT=PROJECT.resolve()
WORK=Path('/content/vmb_collect_all_work')
WORK.mkdir(parents=True,exist_ok=True)
CHECKOUT=Path('/content/VietMedBridge-collect-all')
REPO='https://github.com/Platypus27-coder/VietMedBridge.git'
COMMIT='bbf09ec01d542a5b06a4c2beec245571abeb3515'
if not (CHECKOUT/'.git').is_dir():
    subprocess.run(['git','clone','--no-checkout',REPO,str(CHECKOUT)],check=True)
if subprocess.check_output(['git','-C',str(CHECKOUT),'remote','get-url','origin'],text=True).strip()!=REPO:
    raise ValueError('Unexpected checkout repository.')
subprocess.run(['git','-C',str(CHECKOUT),'fetch','--depth','1','origin',COMMIT],check=True)
subprocess.run(['git','-C',str(CHECKOUT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.', 'numpy'],cwd=CHECKOUT,check=True)
sys.path[:0]=[str(WORK),str(CHECKOUT/'src')]
importlib.invalidate_caches()
auth.authenticate_user()

In [ ]:
#@title 2. Khôi phục SQL checkpoint và mapping đã xử lý — không đọc lại corpus
SOURCE = '"""Collect all available BTC URL content from every AI Guru archive version.\n\nRead metadata members through Drive Range, never unpack raw TARs. Original\nsnapshot/hash provenance is preserved; current BTC membership is a separate join.\nThis collects extraction candidates, not silently promoted chunks or embeddings.\n"""\nfrom contextlib import contextmanager\nfrom pathlib import Path\nimport gzip\nimport hashlib\nimport json\nimport re\nimport shutil\nimport sqlite3\nimport tarfile\nimport tempfile\nimport time\n\nimport duckdb\nimport pyarrow as pa\nimport pyarrow.parquet as pq\nfrom vietmedbridge.artifacts import atomic_json, digest_json, publish_file, read_json, sha256_file, verify_file, utc_now\nfrom vietmedbridge.content_embeddings import checked, seal\nfrom vietmedbridge.external_import import _hold, MEMBERS\nfrom vietmedbridge.isolated_intake import Window\n\nFOLDER = \'1P2ke-yIDwj443R_GMegaiH9RACf-Gzth\'\nOFFICIAL_SHA = \'87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7\'\nCAMPAIGN = \'btc-all-ai-guru-71be0e12-v1\'\n\n\ndef fetch_url(value):\n    return value.strip().split(\'#\', 1)[0]\n\n\ndef inventory(session, folder=FOLDER):\n    result, visited, queue = [], set(), [folder]\n    while queue:\n        current = queue.pop(0)\n        if current in visited:\n            continue\n        visited.add(current)\n        token = None\n        while True:\n            params = {\'q\': f"\'{current}\' in parents and trashed=false", \'pageSize\': 1000,\n                      \'fields\': \'nextPageToken,files(id,name,mimeType,size,modifiedTime,md5Checksum,shortcutDetails)\',\n                      \'supportsAllDrives\': \'true\', \'includeItemsFromAllDrives\': \'true\'}\n            if token:\n                params[\'pageToken\'] = token\n            response = session.get(\'https://www.googleapis.com/drive/v3/files\', params=params, timeout=90)\n            response.raise_for_status()\n            body = response.json()\n            for item in body.get(\'files\', []):\n                if item[\'mimeType\'] == \'application/vnd.google-apps.shortcut\':\n                    target = item[\'shortcutDetails\']\n                    response = session.get(\'https://www.googleapis.com/drive/v3/files/\'+target[\'targetId\'], params={\n                        \'fields\': \'id,name,mimeType,size,modifiedTime,md5Checksum\', \'supportsAllDrives\': \'true\'}, timeout=90)\n                    response.raise_for_status()\n                    item = response.json()\n                if item[\'mimeType\'] == \'application/vnd.google-apps.folder\':\n                    queue.append(item[\'id\'])\n                elif item[\'name\'].lower().endswith(\'.tar\'):\n                    if not item.get(\'size\') or not item.get(\'md5Checksum\'):\n                        raise ValueError(\'Archive has no immutable byte identity.\')\n                    result.append(item)\n            token = body.get(\'nextPageToken\')\n            if not token:\n                break\n    return sorted({r[\'id\']: r for r in result}.values(), key=lambda r: (r.get(\'modifiedTime\', \'\'), r[\'id\']), reverse=True)\n\n\ndef identity(remote):\n    return {k: remote[k] for k in (\'id\', \'name\', \'size\', \'modifiedTime\', \'md5Checksum\')}\n\n\n@contextmanager\ndef metadata_leaves(reader, remote, retained, work):\n    """Read native leaf metadata, or enumerate every child TAR of a container.\n\n    A native source\'s four manifests are its authoritative URL inventory. Its\n    raw members are payloads, not additional sources, and must not be enumerated\n    after the manifests are complete. Containers have no native manifests; walk\n    their complete member list so later resume TARs are still discovered.\n    """\n    with reader.open(remote[\'id\'], identity(remote)) as stream:\n        if hasattr(stream, \'block_size\'):\n            stream.block_size = 65536\n        leaves = []\n        def scan(start, size, name, depth=0):\n            if depth > 3:\n                raise ValueError(\'Nested archive depth exceeds the bounded scanner.\')\n            found, nested = {}, []\n            headers, logged = 0, time.monotonic()\n            with tarfile.open(fileobj=Window(stream, start, size), mode=\'r:\') as archive:\n                for member in archive:\n                    headers += 1\n                    if time.monotonic()-logged >= 15:\n                        print(\'ARCHIVE_HEADER_PROGRESS:\',name,\'|\',headers,\'headers | metadata\',len(found),\'/4\',flush=True)\n                        logged = time.monotonic()\n                    if not member.isfile():\n                        continue\n                    label = (\'dataset\' if member.name == MEMBERS[\'dataset\'] else\n                             \'crawl\' if member.name == MEMBERS[\'crawl\'] else\n                             \'extracted\' if member.name == MEMBERS[\'extracted\'] else\n                             \'frontier\' if re.fullmatch(r\'data/crawl_shards/shard_[0-9]+[.]parquet\', member.name) else None)\n                    if label:\n                        if nested:\n                            raise ValueError(\'Mixed native/container layout; do not choose an arbitrary source.\')\n                        if label in found or not 0 < member.size <= 2*1024**3:\n                            raise ValueError(\'Duplicate/oversize metadata member; no arbitrary version selected.\')\n                        found[label] = {\'path\': member.name, \'offset\': start+member.offset_data,\n                                        \'bytes\': member.size, \'sha256\': member.pax_headers.get(\'vibiomir.sha256\')}\n                        if len(found) == 4:\n                            print(\'NATIVE_METADATA_READY:\',name,\'|\',headers,\n                                  \'headers | 4/4 manifests; raw member listing skipped\',flush=True)\n                            break\n                    elif member.name.endswith(\'.tar\') and not member.name.startswith(\'raw/\'):\n                        if found:\n                            raise ValueError(\'Mixed native/container layout; source requires inspection.\')\n                        nested.append((start+member.offset_data, member.size, name+\'!\'+member.name))\n            if found:\n                if set(found) != {\'dataset\',\'crawl\',\'extracted\',\'frontier\'}:\n                    raise ValueError(\'Incomplete leaf metadata; preserve this archive for inspection.\')\n                leaves.append({\'name\': name, \'format\': \'native\', \'members\': found})\n            for entry in nested:\n                scan(*entry, depth+1)\n        if remote[\'id\'] == retained[\'drive_file_id\']:\n            if int(remote[\'size\']) != retained[\'archive_bytes\']:\n                raise ValueError(\'Pinned retained package changed; do not guess member offsets.\')\n            members = retained[\'members\']\n            for item in members.values():\n                stream.seek(item[\'offset\']-512)\n                header = tarfile.TarInfo.frombuf(stream.read(512), \'utf-8\', \'surrogateescape\')\n                if header.name != item[\'path\'] or header.size != item[\'bytes\'] or not header.isfile():\n                    raise ValueError(\'Retained member header changed.\')\n            leaves.append({\'name\': remote[\'name\'], \'format\': \'retained\', \'members\': members})\n        else:\n            scan(0, int(remote[\'size\']), remote[\'name\'])\n        if not leaves:\n            raise ValueError(\'No supported metadata leaf discovered in this archive.\')\n        yield stream, leaves\n\n\ndef stage_member(stream, item, target):\n    if hasattr(stream, \'block_size\'):\n        stream.block_size = 8*1024**2\n        stream.block_start, stream.block = -1, b\'\'\n    stream.seek(item[\'offset\'])\n    digest, remaining = hashlib.sha256(), item[\'bytes\']\n    logged = time.monotonic()\n    with Path(target).open(\'wb\') as out:\n        while remaining:\n            payload = stream.read(min(8*1024**2, remaining))\n            if not payload:\n                raise EOFError(\'Truncated metadata member.\')\n            out.write(payload)\n            digest.update(payload)\n            remaining -= len(payload)\n            if time.monotonic()-logged>=15 or remaining==0:\n                print(\'METADATA_TRANSFER\',item[\'path\'],\'|\',round((item[\'bytes\']-remaining)/1024**2,1),\n                      \'/\',round(item[\'bytes\']/1024**2,1),\'MiB\',flush=True)\n                logged=time.monotonic()\n    if item.get(\'sha256\') and item[\'sha256\'] != digest.hexdigest():\n        raise ValueError(\'Source metadata checksum differs from the TAR receipt.\')\n    return digest.hexdigest()\n\n\nclass CollectAll:\n    def __init__(self, project, work, *, official_sha=OFFICIAL_SHA):\n        self.project, self.work = Path(project).resolve(), Path(work).resolve()\n        self.root = self.project/\'data_staging\'/CAMPAIGN\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.work.mkdir(parents=True, exist_ok=True)\n        self.binding = {\'schema\': \'whole-folder-content-collection-v1\', \'official_sha256\': official_sha,\n                        \'folder_id\': FOLDER, \'normalization\': \'trim-and-remove-fragment-only\'}\n        self.database = self.work/\'collection.sqlite\'\n        self.pointer = self.root/\'checkpoint.json\'\n        if not self.database.exists() and self.pointer.exists():\n            saved = checked(read_json(self.pointer))\n            if saved[\'binding\'] != self.binding:\n                raise ValueError(\'Collection BTC snapshot changed.\')\n            payload = self.root/saved[\'path\']\n            verify_file(payload, saved[\'sha256\'])\n            print(\'RESTORE_COLLECTION: one SQL checkpoint; no old archives reopened\', flush=True)\n            with gzip.open(payload, \'rb\') as source, self.database.open(\'wb\') as out:\n                shutil.copyfileobj(source, out, length=1024**2)\n            verify_file(self.database, saved[\'sqlite_sha256\'])\n        self.con = sqlite3.connect(self.database)\n        self.con.execute(\'PRAGMA cache_size=-32768\')\n        self.con.execute(\'PRAGMA synchronous=FULL\')\n        self.con.execute(\'CREATE TABLE IF NOT EXISTS contract(value TEXT)\')\n        row = self.con.execute(\'SELECT value FROM contract\').fetchone()\n        if row and row[0] != digest_json(self.binding):\n            raise ValueError(\'Retained local collection belongs to another BTC snapshot.\')\n        if not row:\n            self.con.execute(\'INSERT INTO contract VALUES (?)\', [digest_json(self.binding)])\n        self.con.executescript(\'\'\'\n            CREATE TABLE IF NOT EXISTS official(id INTEGER PRIMARY KEY,url TEXT NOT NULL,fetch TEXT NOT NULL);\n            CREATE INDEX IF NOT EXISTS official_fetch ON official(fetch);\n            CREATE TABLE IF NOT EXISTS content(fetch TEXT PRIMARY KEY,seen INTEGER DEFAULT 0,raw INTEGER DEFAULT 0,\n                existing TEXT,candidate TEXT,reason TEXT);\n            CREATE TABLE IF NOT EXISTS completed(id TEXT PRIMARY KEY,kind TEXT,receipt TEXT);\n        \'\'\')\n        self.con.commit()\n\n    def saved(self, identifier):\n        row = self.con.execute(\'SELECT receipt FROM completed WHERE id=?\', [identifier]).fetchone()\n        return json.loads(row[0]) if row else None\n\n    def commit(self, identifier, kind, receipt):\n        self.con.execute(\'INSERT INTO completed VALUES (?,?,?)\', [identifier, kind, json.dumps(receipt)])\n        self.con.commit()\n        self.snapshot()\n\n    def snapshot(self):\n        copy = self.work/\'backup.sqlite\'\n        destination = sqlite3.connect(copy)\n        try:\n            self.con.backup(destination)\n        finally:\n            destination.close()\n        payload = self.work/\'backup.sqlite.gz\'\n        with copy.open(\'rb\') as source, gzip.open(payload, \'wb\', compresslevel=1) as out:\n            shutil.copyfileobj(source, out, length=1024**2)\n        checksum = sha256_file(payload)\n        target = self.root/(\'sql-\'+checksum+\'.gz\')\n        publish_file(payload, target)\n        atomic_json(self.pointer, seal({\'binding\': self.binding, \'path\': target.name, \'sha256\': checksum,\n            \'sqlite_sha256\': sha256_file(copy), \'updated_at\': utc_now()}))\n        copy.unlink()\n        payload.unlink()\n        print(\'COLLECTION_SQL_CHECKPOINT_SAVED_ON_DRIVE\', flush=True)\n\n    def official(self, path):\n        if self.saved(\'official\'):\n            return\n        verify_file(path, self.binding[\'official_sha256\'])\n        try:\n            for batch in pq.ParquetFile(path).iter_batches(batch_size=8192, columns=[\'id\',\'url\']):\n                rows = [(r[\'id\'], r[\'url\'], fetch_url(r[\'url\'])) for r in batch.to_pylist()]\n                self.con.executemany(\'INSERT INTO official VALUES (?,?,?)\', rows)\n                self.con.executemany(\'INSERT OR IGNORE INTO content(fetch) VALUES (?)\', [(r[2],) for r in rows])\n            self.commit(\'official\', \'reference\', {\'sha256\': self.binding[\'official_sha256\']})\n        except BaseException:\n            self.con.rollback()\n            raise\n\n    def existing(self, path, checksum, label):\n        identifier = \'existing:\'+checksum\n        if self.saved(identifier):\n            print(\'SKIP_EXISTING_ALIAS_MAPPING:\', label, flush=True)\n            return\n        verify_file(path, checksum)\n        try:\n            for batch in pq.ParquetFile(path).iter_batches(batch_size=8192, columns=[\'doc_id\',\'url\']):\n                rows = [(json.dumps({\'mapping\': label, \'original_doc_id\': r[\'doc_id\'], \'original_url\': r[\'url\']}), fetch_url(r[\'url\']))\n                        for r in batch.to_pylist()]\n                self.con.executemany(\'UPDATE content SET existing=coalesce(existing,?) WHERE fetch=?\', rows)\n            self.commit(identifier, \'existing\', {\'mapping\': label, \'sha256\': checksum})\n        except BaseException:\n            self.con.rollback()\n            raise\n\n    def leaf(self, stream, remote, leaf, *, version_check=None):\n        identifier = digest_json({\'remote\': identity(remote), \'leaf\': leaf})\n        if self.saved(identifier):\n            print(\'SKIP_COMPLETED_ARCHIVE_LEAF:\', leaf[\'name\'], \'| receipt only\', flush=True)\n            return\n        print(\'COLLECT_ARCHIVE_LEAF:\', leaf[\'name\'], \'| metadata only, no raw TAR copy\', flush=True)\n        with tempfile.TemporaryDirectory(prefix=\'metadata-\', dir=self.work) as temporary:\n            local = Path(temporary)\n            staged_checksums={}\n            for kind, item in leaf[\'members\'].items():\n                if leaf[\'format\'] == \'retained\' and kind == \'mapping\':\n                    continue\n                print(\'READ_METADATA\', kind, \'|\', round(item[\'bytes\']/1024**2, 1), \'MiB\', flush=True)\n                staged_checksums[kind]=stage_member(stream,item,local/(kind+\'.\'+(\'json\' if kind == \'dataset\' else \'parquet\')))\n            descriptor = read_json(local/\'dataset.json\')\n            con = duckdb.connect()\n            con.execute("SET memory_limit=\'1GB\'")\n            con.execute(\'SET threads=1\')\n            con.execute(\'SET preserve_insertion_order=false\')\n            con.execute(\'SET temp_directory=?\', [str(local/\'spill\')])\n            try:\n                if leaf[\'format\'] == \'native\':\n                    for kind in (\'frontier\',\'crawl\',\'extracted\'):\n                        con.read_parquet(str(local/(kind+\'.parquet\'))).create_view(kind)\n                        duplicates=con.execute(f\'SELECT count(*)-count(DISTINCT crawl_url_id) FROM {kind}\').fetchone()[0]\n                        if duplicates:\n                            raise ValueError(\'Duplicate crawl_url_id in source metadata; no arbitrary row selected.\')\n                    query = \'\'\'SELECT f.fetch_url,c.status,c.raw_sha256,c.raw_path,c.raw_size_bytes,c.final_url,\n                        c.content_type,c.snapshot_id AS crawl_snapshot,e.snapshot_id AS extract_snapshot,\n                        e.raw_sha256 AS extract_raw_sha,e.extract_status,e.text,e.title,e.char_count,\n                        to_json(c) AS crawl_json,to_json(e) AS extracted_json\n                        FROM frontier f LEFT JOIN crawl c USING(crawl_url_id) LEFT JOIN extracted e USING(crawl_url_id)\'\'\'\n                else:\n                    con.read_parquet(str(local/\'raw_catalog.parquet\')).create_view(\'raw\')\n                    con.read_parquet(str(local/\'documents.parquet\')).create_view(\'docs\')\n                    for kind in (\'raw\',\'docs\'):\n                        if con.execute(f\'SELECT count(*)-count(DISTINCT crawl_url_id) FROM {kind}\').fetchone()[0]:\n                            raise ValueError(\'Duplicate retained crawl_url_id; source requires inspection.\')\n                    query = \'\'\'SELECT r.fetch_url,\'SUCCESS_RAW\' AS status,r.raw_sha256,r.raw_path,r.raw_size_bytes,r.final_url,\n                        r.content_type,r.original_snapshot_id AS crawl_snapshot,e.snapshot_id AS extract_snapshot,\n                        e.raw_sha256 AS extract_raw_sha,e.extract_status,e.text,e.title,e.char_count,\n                        r.original_crawl_record_json AS crawl_json,to_json(e) AS extracted_json,r.quality_tier AS raw_quality_tier\n                        FROM raw r LEFT JOIN docs e USING(crawl_url_id)\'\'\'\n                output = local/\'recovered.parquet\'\n                schema = pa.schema([(\'fetch_url\',pa.string()),(\'source_file_id\',pa.string()),(\'leaf\',pa.string()),\n                    (\'original_snapshot_id\',pa.string()),(\'source_text_sha256\',pa.string()),\n                    (\'crawl_json\',pa.string()),(\'extracted_json\',pa.large_string())])\n                count, matched, raw_count, held = 0, 0, 0, {}\n                logged=time.monotonic()\n                with pq.ParquetWriter(output, schema, compression=\'zstd\') as writer:\n                    for batch in con.execute(query).fetch_record_batch(128):\n                        candidates = []\n                        for row in batch.to_pylist():\n                            url = row[\'fetch_url\']\n                            state = self.con.execute(\'SELECT existing,candidate FROM content WHERE fetch=?\', [url]).fetchone()\n                            if state is None:\n                                continue\n                            matched += 1\n                            original = row[\'crawl_snapshot\']\n                            raw_ok = (row[\'status\'] == \'SUCCESS_RAW\' and isinstance(row[\'raw_sha256\'],str)\n                                and re.fullmatch(\'[0-9a-f]{64}\', row[\'raw_sha256\']) and row[\'raw_path\']\n                                and row[\'raw_path\'].startswith(\'raw/\') and not \'..\' in Path(row[\'raw_path\']).parts and original)\n                            raw_count += bool(raw_ok)\n                            self.con.execute(\'UPDATE content SET seen=1,raw=max(raw,?) WHERE fetch=?\', [int(bool(raw_ok)),url])\n                            if state[0] is not None or state[1] is not None:\n                                continue\n                            reason = None\n                            if not raw_ok:\n                                reason = row[\'status\'] or \'MISSING_CRAWL_RESULT\'\n                            elif leaf[\'format\'] == \'native\' and original != descriptor.get(\'snapshot_id\'):\n                                reason = \'ARCHIVE_SNAPSHOT_BINDING_MISMATCH\'\n                            elif row[\'extract_raw_sha\'] != row[\'raw_sha256\'] or row[\'extract_snapshot\'] != original:\n                                reason = \'MISSING_OR_UNBOUND_EXTRACTION\'\n                            elif row[\'extract_status\'] == \'EXTRACT_SUCCESS\' and len(row[\'text\'] or \'\') != row[\'char_count\']:\n                                reason = \'INVALID_SOURCE_LENGTH\'\n                            else:\n                                extra = json.loads(row[\'extracted_json\']) if row[\'extracted_json\'] else {}\n                                gate_row = {**row, \'url\':url, \'source_issue\':extra.get(\'source_issue\')}\n                                hold = _hold(gate_row, 2_000_000)\n                                reason = \':\'.join(hold) if hold else None\n                                if leaf[\'format\'] == \'retained\' and any(str(t).upper() in (\'LOW\',\'QUARANTINE\')\n                                    for t in (extra.get(\'quality_tier\'),row.get(\'raw_quality_tier\'))):\n                                    reason = \'RETAINED_QUALITY_REVIEW\'\n                            if reason:\n                                held[reason] = held.get(reason,0)+1\n                                self.con.execute(\'UPDATE content SET reason=? WHERE fetch=? AND candidate IS NULL\', [reason,url])\n                                continue\n                            reference = {\'path\': \'parts/\'+identifier+\'.parquet\', \'row\':count,\'receipt_id\':identifier,\n                                         \'source_file_id\':remote[\'id\'], \'leaf\':leaf[\'name\'], \'original_snapshot_id\':original}\n                            candidates.append({\'fetch_url\':url,\'source_file_id\':remote[\'id\'],\'leaf\':leaf[\'name\'],\n                                \'original_snapshot_id\':original,\'source_text_sha256\':hashlib.sha256(row[\'text\'].encode()).hexdigest(),\n                                \'crawl_json\':row[\'crawl_json\'],\'extracted_json\':row[\'extracted_json\']})\n                            self.con.execute(\'UPDATE content SET candidate=?,reason=NULL WHERE fetch=?\', [json.dumps(reference),url])\n                            count += 1\n                        if candidates:\n                            writer.write_table(pa.Table.from_pylist(candidates,schema=schema))\n                        if time.monotonic()-logged>=15:\n                            print(\'COLLECT_PROGRESS\',leaf[\'name\'],\'| matched\',matched,\'| recovery candidates\',count,flush=True)\n                            logged=time.monotonic()\n                target = self.root/\'parts\'/(identifier+\'.parquet\')\n                if version_check is not None:\n                    version_check()\n                checksum = publish_file(output,target)\n                self.commit(identifier,\'leaf\',{\'archive\':identity(remote),\'leaf\':leaf,\'candidates\':count,\n                    \'matched_rows\':matched,\'raw_rows\':raw_count,\'held\':held,\'path\':target.relative_to(self.root).as_posix(),\n                    \'sha256\':checksum,\'metadata_sha256\':staged_checksums,\'original_snapshot_provenance_preserved\':True})\n                print(\'COLLECT_LEAF_SAVED:\', count, \'new candidate URLs |\', leaf[\'name\'], flush=True)\n            except BaseException:\n                self.con.rollback()\n                raise\n            finally:\n                con.close()\n\n    def report(self, files, errors):\n        # Count official IDs and distinct fetch URLs separately. Failed records never count as usable text.\n        state = "CASE WHEN c.existing IS NOT NULL THEN \'REUSED_VALIDATED_CONTENT\' WHEN c.candidate IS NOT NULL THEN \'RECOVERED_EXTRACTION_PENDING_CHUNK\' WHEN c.seen=0 THEN \'NOT_FOUND_IN_ARCHIVES\' WHEN c.raw=1 THEN \'RAW_AVAILABLE_NEEDS_EXTRACTION\' ELSE \'CRAWL_NOT_SUCCESSFUL\' END"\n        query = f\'SELECT o.id,o.url,o.fetch,{state} AS state,c.existing,c.candidate,c.reason FROM official o JOIN content c ON o.fetch=c.fetch\'\n        counts = dict(self.con.execute(f\'SELECT state,count(*) FROM ({query}) GROUP BY state\'))\n        total = self.con.execute(\'SELECT count(*) FROM official\').fetchone()[0]\n        unique = self.con.execute(\'SELECT count(*) FROM content\').fetchone()[0]\n        seen,raw=self.con.execute(\'SELECT sum(c.seen),sum(c.raw) FROM official o JOIN content c ON o.fetch=c.fetch\').fetchone()\n        files_out={}\n        for name, sql in [(\'btc_membership.csv\',query),(\'missing_or_unusable.csv\',f"SELECT * FROM ({query}) WHERE state NOT IN (\'REUSED_VALIDATED_CONTENT\',\'RECOVERED_EXTRACTION_PENDING_CHUNK\')")]:\n            import csv\n            path = self.work/name\n            cursor = self.con.execute(sql)\n            with path.open(\'w\',encoding=\'utf-8\',newline=\'\') as out:\n                writer = csv.writer(out)\n                writer.writerow([c[0] for c in cursor.description])\n                while rows := cursor.fetchmany(8192):\n                    writer.writerows(rows)\n            files_out[name]={\'path\':name,\'sha256\':publish_file(path,self.root/name)}\n        result = seal({\'state\':\'ALL_FOLDER_SOURCES_COLLECTED\' if not errors else \'FOLDER_COLLECTION_INCOMPLETE\',\n            \'official_ids\':total,\'official_unique_fetch_urls\':unique,\'counts_by_official_id\':counts,\n            \'official_ids_seen_in_archive_metadata\':seen,\'official_ids_with_successful_raw_record\':raw,\n            \'all_official_ids_represented_in_archives\':not errors and seen==total,\n            \'all_official_raw_records_successful\':not errors and raw==total,\n            \'archive_files\':len(files),\'archive_errors\':errors,\'files\':files_out,\n            \'recovery_parts\':[json.loads(r[0]) for r in self.con.execute("SELECT receipt FROM completed WHERE kind=\'leaf\'")],\n            \'all_official_content_available\':not errors and sum(counts.get(k,0) for k in (\'REUSED_VALIDATED_CONTENT\',\'RECOVERED_EXTRACTION_PENDING_CHUNK\'))==total,\n            \'recovered_text_requires_standard_chunk_pipeline\':True,\'raw_payloads_rehashed\':False,\n            \'submission_candidate_changed\':False,\'updated_at\':utc_now()})\n        atomic_json(self.root/\'collection_report.json\',result)\n        return result\n\n    def run(self, reader, files, retained):\n        errors = []\n        for index, remote in enumerate(files,1):\n            print(f\'ALL_AI_GURU {index}/{len(files)} | {remote["name"]}\',flush=True)\n            archive_id = \'archive:\'+digest_json(identity(remote))\n            if self.saved(archive_id):\n                print(\'SKIP_COMPLETED_ARCHIVE | durable SQL receipt; no TAR reopened\',flush=True)\n                continue\n            try:\n                def version_check():\n                    if identity(reader.metadata(remote[\'id\'])) != identity(remote):\n                        raise ValueError(\'Archive changed during collection; this leaf was not committed.\')\n                with metadata_leaves(reader,remote,retained,self.work) as (stream,leaves):\n                    for leaf in leaves:\n                        self.leaf(stream,remote,leaf,version_check=version_check)\n                if identity(reader.metadata(remote[\'id\'])) != identity(remote):\n                    raise ValueError(\'Archive changed during collection.\')\n                self.commit(archive_id,\'archive\',{\'remote\':identity(remote),\'leaves\':len(leaves)})\n            except Exception as error:\n                self.con.rollback()\n                errors.append({\'file\':remote[\'name\'],\'id\':remote[\'id\'],\'error\':str(error)[:500]})\n                print(\'ARCHIVE_NEEDS_INSPECTION:\',remote[\'name\'],type(error).__name__,str(error)[:200],flush=True)\n        return self.report(files,errors)\n\n    def close(self):\n        self.con.close()\n'
RETAINED = json.loads('{"job_id": "retained-71be0e12", "mode": "NEW_SNAPSHOT_SOURCE", "format": "RETAINED_PORTABLE_V1", "archive": "vibiomir_retained_71be0e12.tar", "drive_file_id": "1Hf6cb7Des8AhYkyWXE3d3PvnDT-BMMQ-", "archive_bytes": 12581928960, "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "original_snapshot_id": "b915d5d5d640d497d103fa37a5a047c1d093772ab17b09638634de4bbc80e5d7", "official_links_sha256": "87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7", "members": {"mapping": {"path": "data/doc_url_map.parquet", "offset": 12562358272, "bytes": 19545473, "sha256": "1dc06041fded26cfe2dfa0c64e39875840a8b3a8051ac58080608497fadaee01"}, "raw_catalog": {"path": "data/raw_catalog.parquet", "offset": 11313683456, "bytes": 125719140, "sha256": "90e5ae894488dd7e47a76dd4664bcaa8156df72245f18c078db1b66ac0f4f9e9"}, "documents": {"path": "data/documents.parquet", "offset": 11439404544, "bytes": 1122951853, "sha256": "b031ec37043f1de3e6f24a70e508fd3e569fb8dff319a59c69489a3bd8adabc3"}, "dataset": {"path": "new_dataset_manifest.json", "offset": 12581909504, "bytes": 1093, "sha256": "912ac3c910985b5f3f003f3d90ea317b13c9c568da7baf424d87488b1cfe7d31"}}}')

(WORK/'collect_all_btc_sources.py').write_text(SOURCE,encoding='utf-8')
import collect_all_btc_sources as collector
# Re-uploading a repaired notebook must replace the old imported reader
# even when the user retains the same Colab runtime and Drive checkpoint.
collector=importlib.reload(collector)
from vietmedbridge.isolated_intake import DriveSource,fetch_reference
from vietmedbridge.artifacts import read_json,verify_file,digest_json
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
session=AuthorizedSession(credentials)
reader=DriveSource(session)
flow=collector.CollectAll(PROJECT,WORK)
scope=read_json(PROJECT/parent/'full_intake_scope.json')
# Only membership/mapping files are read; no old document text, vectors, or index.
if not flow.saved('official'):
    links=PROJECT/parent/'references/5586f423916005c1-links.parquet'
    flow.official(links)
mappings=[(PROJECT/parent/'references/baseline_aliases.parquet',scope['base_plan']['baseline']['sha256'])]
batch_root=PROJECT/parent/'freeze_batch_v1'
batch=read_json(batch_root/'freeze_batch.json')
collector.checked(batch)
for entry in batch['builds']:
    build=(PROJECT/entry['build_dir']).resolve()
    if not build.is_relative_to(PROJECT):
        raise ValueError('Build mapping escapes project.')
    candidate=read_json(build/entry['candidate_name'])
    sha=candidate['candidate_manifest_sha256']
    if sha!=entry['candidate_manifest_sha256'] or digest_json({k:v for k,v in candidate.items() if k!='candidate_manifest_sha256'})!=sha:
        raise ValueError('Frozen mapping producer changed.')
    item=candidate['health']['files']['canonical_aliases']
    path=(build/item['path']).resolve()
    if not path.is_relative_to(build):
        raise ValueError('Alias mapping escapes build.')
    mappings.append((path,item['sha256']))
for path,checksum in mappings:
    # A committed mapping is not opened/rehashed again in another runtime.
    if not flow.saved('existing:'+checksum):
        flow.existing(path,checksum,path.relative_to(PROJECT).as_posix())
print('EXISTING_MAPPING_READY: baseline + all 10 frozen sources retained.',flush=True)


In [ ]:
#@title 3. Thu gom tất cả archive và xuất danh sách URL BTC còn thiếu/thất bại
files=collector.inventory(session)
print('AI_GURU_PHYSICAL_ARCHIVES:',len(files),flush=True)
print('Đọc mọi bản resume; không bỏ archive theo số shard/ngày hoặc snapshot cũ.',flush=True)
try:
    REPORT=flow.run(reader,files,RETAINED)
    print(json.dumps(REPORT,ensure_ascii=False,indent=2),flush=True)
    print('Kết quả:',flow.root,flush=True)
    print('btc_membership.csv: mỗi ID BTC có trạng thái và provenance.',flush=True)
    print('missing_or_unusable.csv: chỉ các URL chưa có nội dung dùng được.',flush=True)
    print('parts/: nội dung khôi phục từ archive; cần đi qua chunk/freeze chuẩn trước embedding.',flush=True)
    if REPORT['archive_errors']:
        raise RuntimeError('Có archive chưa đọc được; không kết luận đủ BTC. Các source đã hoàn tất vẫn có checkpoint.')
finally:
    flow.close()